<a href="https://colab.research.google.com/github/aartinangare1745/GuardPrompts/blob/main/Model_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df=pd.read_csv("df.csv")

In [ ]:
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import numpy as np

# stratify by label + source together
df["strata"] = df["label"].astype(str) + "_" + df["source"]

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["strata"]
)

print(train_df.groupby("source")["label"].value_counts(normalize=True))
print(test_df.groupby("source")["label"].value_counts(normalize=True))

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

# embeddings — run this part on Colab if the dataset feels slow locally
model = SentenceTransformer("all-MiniLM-L6-v2")

X_train = model.encode(train_df["prompt"].tolist(), batch_size=32, show_progress_bar=True)
X_test = model.encode(test_df["prompt"].tolist(), batch_size=32, show_progress_bar=True)

np.save("X_train_emb.npy", X_train)
np.save("X_test_emb.npy", X_test)
np.save("y_train.npy", train_df["label"].values)
np.save("y_test.npy", test_df["label"].values)

source    label
Deepset   0        0.633803
          1        0.366197
MindGard  1        1.000000
NeCent    0        0.723649
          1        0.276351
Name: proportion, dtype: float64
source    label
Deepset   0        0.632075
          1        0.367925
MindGard  1        1.000000
NeCent    0        0.723599
          1        0.276401
Name: proportion, dtype: float64


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/889 [00:00<?, ?it/s]

Batches:   0%|          | 0/223 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score,roc_auc_score
import json




In [ ]:
import os

# Create a directory to save the models if it doesn't exist
output_dir = 'models'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
X_train = np.load("X_train_emb.npy")
X_test = np.load("X_test_emb.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")
test_df = pd.read_csv("test.csv")  # for per-source breakdown

results = {}

for name, model in [
    ("logreg", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ("rf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
]:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(y_test, y_pred, output_dict=True)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {"report": report, "roc_auc": auc}
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", auc)

    joblib.dump(model, f"models/{name}.joblib")

    # per-source breakdown — this is your key finding
    test_df["pred"] = y_pred
    for src in test_df["source"].unique():
        mask = test_df["source"] == src
        f1 = f1_score(y_test[mask], y_pred[mask])
        print(f"  {src}: F1 = {f1:.3f}  (n={mask.sum()})")

with open("metrics.json", "w") as f:
    json.dump(results, f, indent=2, default=str)


--- logreg ---
              precision    recall  f1-score   support

           0       0.92      0.87      0.89      4329
           1       0.81      0.88      0.84      2781

    accuracy                           0.87      7110
   macro avg       0.86      0.87      0.87      7110
weighted avg       0.88      0.87      0.87      7110

ROC-AUC: 0.9423188020814776
  NeCent: F1 = 0.770  (n=5890)
  MindGard: F1 = 0.974  (n=1114)
  Deepset: F1 = 0.462  (n=106)

--- rf ---
              precision    recall  f1-score   support

           0       0.86      0.97      0.91      4329
           1       0.93      0.75      0.83      2781

    accuracy                           0.88      7110
   macro avg       0.89      0.86      0.87      7110
weighted avg       0.89      0.88      0.88      7110

ROC-AUC: 0.9666737104709058
  NeCent: F1 = 0.753  (n=5890)
  MindGard: F1 = 0.948  (n=1114)
  Deepset: F1 = 0.098  (n=106)
